Checkpoint 2:

In [1]:
import pandas as pd
df = pd.read_csv("room_amenities_cleaned.csv")
df.drop(columns=["amenity_group"], inplace=True)

In [2]:
df

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
10061,9367,[]
10062,9742,[]
10063,9743,[]
10064,9749,[]


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10066 entries, 0 to 10065
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   room_type_id    10066 non-null  int64 
 1   room_amenities  10066 non-null  object
dtypes: int64(1), object(1)
memory usage: 157.4+ KB


In [4]:
import ast
def extract_bed_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bed":
           return amenity.get("amenity")
    return None

In [5]:

df["bed_str"] = df["room_amenities"].apply(extract_bed_features)

In [6]:
df

,room_type_id,room_amenities,bed_str
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn
...,...,...,...
10061,9367,[],None
10062,9742,[],None
10063,9743,[],None
10064,9749,[],None


In [7]:
df["bed_str"] = df["bed_str"].str.replace(r"/", "&")
df["bed_str"] = df["bed_str"].str.replace(r"và", "&")

In [8]:
df["expanded_bed"] = df["bed_str"].str.split("&|hoặc").apply(lambda x: [item.strip() for item in x] if isinstance(x, list) else x)

In [9]:
import re

def process_expanded_bed(bed_list):
    if not isinstance(bed_list, list):
        return []
    processed_beds = []
    for bed in bed_list:
        if not isinstance(bed, str):
            continue
        # Tách các lựa chọn "hoặc"
        for part in re.split(r"hoặc", bed):
            part = part.strip()
            # Tìm số lượng và loại giường
            match = re.match(r"(\d+)\s*(.*)", part)
            if match:
                bed_count = int(match.group(1))
                bed_type = match.group(2).strip()
            else:
                bed_count = 1
                bed_type = part
            if bed_type:
                processed_beds.append({'bed_type': bed_type, 'bed_count': bed_count})
    return processed_beds

In [10]:
df["expanded_bed"] = df["expanded_bed"].apply(process_expanded_bed)

In [11]:
df

,room_type_id,room_amenities,bed_str,expanded_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]"
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]"
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]"
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]"
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1..."
...,...,...,...,...
10061,9367,[],None,[]
10062,9742,[],None,[]
10063,9743,[],None,[]
10064,9749,[],None,[]


In [12]:
bed_values = {
    'giường đôi lớn': "large_double_bed",
    'giường lớn': "large_bed",
    'giường đơn' : "single_bed",
    'giường sofa': "sofa_bed",
    'giường đôi' : "double_bed",
    'giường đôi nhỏ': "small_double_bed",
    'giường siêu lớn': "king_size_bed",
    'nệm futon': "futon_mattress",
    'giường tầng': "bunk_bed",
    'Giường cực dài': "extra_long_bed"
}

In [13]:
def extract_bed_large_double_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đôi lớn":
            return 1
    return 0

df["large_double_bed"] = df["expanded_bed"].apply(extract_bed_large_double_bed)

In [ ]:
def extract_bed_large_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường lớn":
            return 1
    return 0
df["large_bed"] = df["expanded_bed"].apply(extract_bed_large_bed)

In [15]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,large_double_bed,large_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",1,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",0,1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",0,1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",0,1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...",1,0
...,...,...,...,...,...,...
10061,9367,[],None,[],0,0
10062,9742,[],None,[],0,0
10063,9743,[],None,[],0,0
10064,9749,[],None,[],0,0


In [16]:
def extract_bed_single_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đơn":
            return 1
    return 0

df["single_bed"] = df["expanded_bed"].apply(extract_bed_single_bed)

In [17]:
def extract_bed_sofa_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường sofa":
            return 1
    return 0

df["sofa_bed"] = df["expanded_bed"].apply(extract_bed_sofa_bed)

In [19]:
def extract_bed_double_bed(expanded_bed):
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đôi":
            return 1
    return 0

df["double_bed"] = df["expanded_bed"].apply(extract_bed_double_bed)

In [20]:
def extract_bed_small_double_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "giường đôi nhỏ":
            return 1
    return 0

df["small_double_bed"] = df["expanded_bed"].apply(extract_bed_small_double_bed)

In [21]:
def extract_bed_king_size_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "giường siêu lớn":
            return 1
    return 0

df["king_size_bed"] = df["expanded_bed"].apply(extract_bed_king_size_bed)

In [22]:
def extract_bed_futon_mattress(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "nệm futon":
            return 1
    return 0

df["futon_mattress"] = df["expanded_bed"].apply(extract_bed_futon_mattress)

In [23]:
def extract_bed_bunk_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "giường tầng":
            return 1
    return 0

df["bunk_bed"] = df["expanded_bed"].apply(extract_bed_bunk_bed)

In [24]:
def extract_bed_extra_long_bed(expanded_bed):
    res = 0
    for bed in expanded_bed:
        if bed['bed_type'] == "Giường cực dài":
            return 1
    return 0

df["extra_long_bed"] = df["expanded_bed"].apply(extract_bed_extra_long_bed)

In [26]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,extra_long_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",1,0,0,0,0,0,0,0,0,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",0,1,0,0,0,0,0,0,0,0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",0,1,0,0,0,0,0,0,0,0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",0,1,0,0,0,0,0,0,0,0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...",1,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,9367,[],None,[],0,0,0,0,0,0,0,0,0,0
10062,9742,[],None,[],0,0,0,0,0,0,0,0,0,0
10063,9743,[],None,[],0,0,0,0,0,0,0,0,0,0
10064,9749,[],None,[],0,0,0,0,0,0,0,0,0,0


In [27]:
df.to_csv("checkpoint3.csv", index=False)

In [2]:
import pandas as pd
df = pd.read_csv("checkpoint3.csv")

In [3]:
df

,room_type_id,room_amenities,bed_str,expanded_bed,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,extra_long_bed
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1 giường đôi lớn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1}]",1,0,0,0,0,0,0,0,0,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",0,1,0,0,0,0,0,0,0,0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",2 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 2}]",0,1,0,0,0,0,0,0,0,0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1 giường lớn,"[{'bed_type': 'giường lớn', 'bed_count': 1}]",0,1,0,0,0,0,0,0,0,0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1 giường đôi lớn hoặc 2 giường đơn,"[{'bed_type': 'giường đôi lớn', 'bed_count': 1...",1,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,9367,[],NaN,[],0,0,0,0,0,0,0,0,0,0
10062,9742,[],NaN,[],0,0,0,0,0,0,0,0,0,0
10063,9743,[],NaN,[],0,0,0,0,0,0,0,0,0,0
10064,9749,[],NaN,[],0,0,0,0,0,0,0,0,0,0


In [14]:
def calculate_flexibility_score(bed_str):
    # đếm số lần xuất hiện từ "hoặc"
    return bed_str.count("hoặc") if isinstance(bed_str, str) else 0
df["flexibility_score"] = df["bed_str"].apply(calculate_flexibility_score) + 1

In [18]:
df[["room_type_id","room_amenities", "flexibility_score"]]

,room_type_id,room_amenities,flexibility_score
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",2
...,...,...,...
10061,9367,[],1
10062,9742,[],1
10063,9743,[],1
10064,9749,[],1


In [17]:
df.drop(columns=["expanded_bed", "bed_str"]).to_csv("checkpoint4.csv", index=False)

In [2]:
import pandas as pd
df = pd.read_csv("checkpoint4.csv")

In [3]:
df

,room_type_id,room_amenities,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,extra_long_bed,flexibility_score
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1,0,0,0,0,0,0,0,0,0,1
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1,0,1,0,0,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,9367,[],0,0,0,0,0,0,0,0,0,0,1
10062,9742,[],0,0,0,0,0,0,0,0,0,0,1
10063,9743,[],0,0,0,0,0,0,0,0,0,0,1
10064,9749,[],0,0,0,0,0,0,0,0,0,0,1


In [4]:
import ast
def extract_sqm_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "sqm":
           return amenity.get("amenity")
    return None

df["sqm_str"] = df["room_amenities"].apply(extract_sqm_features)

In [5]:
df

,room_type_id,room_amenities,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,extra_long_bed,flexibility_score,sqm_str
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1,0,0,0,0,0,0,0,0,0,1,Diện tích phòng: 18 m²
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1,Diện tích phòng: 30 m²
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1,Diện tích phòng: 45 m²
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1,Diện tích phòng: 20 m²
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1,0,1,0,0,0,0,0,0,0,2,Diện tích phòng: 35 m²
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,9367,[],0,0,0,0,0,0,0,0,0,0,1,None
10062,9742,[],0,0,0,0,0,0,0,0,0,0,1,None
10063,9743,[],0,0,0,0,0,0,0,0,0,0,1,None
10064,9749,[],0,0,0,0,0,0,0,0,0,0,1,None


In [7]:
import re
def extract_room_area(text):
    """
    Trích xuất diện tích phòng (số nguyên, đơn vị m²) từ chuỗi.
    Ví dụ: "25 m²" hoặc "Diện tích phòng: 20 m²" -> 25 hoặc 20
    """
    if not isinstance(text, str):
        return None
    match = re.search(r'(\d+)\s*m²', text)
    if match:
        return int(match.group(1))
    return None

In [8]:
df["sqm"] = df["sqm_str"].apply(extract_room_area)

In [10]:
df.drop(columns=["sqm_str"], inplace=True)

In [11]:
df

,room_type_id,room_amenities,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,extra_long_bed,flexibility_score,sqm
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1,0,0,0,0,0,0,0,0,0,1,18.0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1,30.0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1,45.0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",0,1,0,0,0,0,0,0,0,0,1,20.0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1,0,1,0,0,0,0,0,0,0,2,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,9367,[],0,0,0,0,0,0,0,0,0,0,1,NaN
10062,9742,[],0,0,0,0,0,0,0,0,0,0,1,NaN
10063,9743,[],0,0,0,0,0,0,0,0,0,0,1,NaN
10064,9749,[],0,0,0,0,0,0,0,0,0,0,1,NaN


In [12]:
df.to_csv("checkpoint5.csv", index=False)

In [ ]:
df = pd.read_csv("checkpoint5.csv")

In [13]:
import ast
def extract_bathrooms_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bathrooms":
           return amenity.get("amenity")
    return None

df["bathrooms_str"] = df["room_amenities"].apply(extract_bathrooms_features)

In [16]:
df[["room_type_id","room_amenities", "bathrooms_str"]]

,room_type_id,room_amenities,bathrooms_str
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",phòng tắm riêng
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",phòng tắm riêng
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",None
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",phòng tắm riêng
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",phòng tắm riêng
...,...,...,...
10061,9367,[],None
10062,9742,[],None
10063,9743,[],None
10064,9749,[],None


In [15]:
df[["bathrooms_str"]].drop_duplicates()

,bathrooms_str
0,phòng tắm riêng
2,None
9,2 phòng tắm
66,3 phòng tắm
84,Phòng tắm chung
165,12 phòng tắm
266,1 phòng tắm
337,5 phòng tắm
533,10 phòng tắm
594,7 phòng tắm


In [19]:
import re
import pandas as pd

def process_bathroom_info(text):
    if not isinstance(text, str):
        # Trường hợp None, giả sử là 0 phòng tắm và không xác định loại
        return pd.Series([0, 1]) 
    
    text_lower = text.lower()
    
    # 1. Xử lý số lượng phòng tắm
    count = 0
    # Tìm số trong chuỗi (ví dụ: "2 phòng tắm")
    match = re.search(r'(\d+)\s*phòng tắm', text_lower)
    if match:
        count = int(match.group(1))
    # Các trường hợp mô tả chữ -> gán là 1
    elif "phòng tắm" in text_lower: 
        count = 1
        
    # 2. Xử lý loại phòng tắm (Riêng tư hay Chung)
    # Mặc định là riêng tư (1), nếu thấy chữ "chung" thì là (0)
    is_private = 1
    if "chung" in text_lower:
        is_private = 0
        
    return pd.Series([count, is_private])

# Áp dụng vào DataFrame
df[["bathroom_count", "is_private_bathroom"]] = df["bathrooms_str"].apply(process_bathroom_info)

In [20]:

# Kiểm tra kết quả
df[["bathrooms_str", "bathroom_count", "is_private_bathroom"]].drop_duplicates()

,bathrooms_str,bathroom_count,is_private_bathroom
0,phòng tắm riêng,1,1
2,None,0,1
9,2 phòng tắm,2,1
66,3 phòng tắm,3,1
84,Phòng tắm chung,1,0
165,12 phòng tắm,12,1
266,1 phòng tắm,1,1
337,5 phòng tắm,5,1
533,10 phòng tắm,10,1
594,7 phòng tắm,7,1


In [22]:
df.drop(columns=["bathrooms_str"], inplace=True)

In [23]:
df.to_csv("checkpoint6.csv", index=False)

In [ ]:
import pandas as pd
df = pd.read_csv("checkpoint4.csv")

In [24]:
import ast
def extract_views_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "views":
           return amenity.get("amenity")
    return None

df["views_str"] = df["room_amenities"].apply(extract_views_features)
df[["views_str"]]

,views_str
0,Hướng Ngoài trời
1,Hướng Núi
2,Hướng Núi
3,Hướng Ngoài trời
4,Hướng Thành phố
...,...
10061,None
10062,None
10063,None
10064,None


In [25]:
df[["views_str"]].drop_duplicates()

,views_str
0,Hướng Ngoài trời
1,Hướng Núi
4,Hướng Thành phố
10,Hướng Không có cửa sổ
11,Hướng Đường phố
12,None
29,Hướng Công viên
45,Hướng Bể bơi
52,Hướng Nông thôn
54,Hướng Vườn


In [33]:
def normalize_views(views_str):
    import unicodedata
    if isinstance(views_str, str):
        # Xóa "Hướng" và strip
        s = views_str.replace("Hướng", "").strip()
        # Chuyển sang không dấu
        s = unicodedata.normalize('NFKD', s)
        s = ''.join([c for c in s if not unicodedata.combining(c)])
        s = s.replace('Đ', 'D').replace('đ', 'd')
        return s.lower()
    return None

In [34]:
df[["views_str"]].drop_duplicates()["views_str"].apply(normalize_views)

0                          ngoai troi
1                                 nui
4                           thanh pho
10                    khong co cua so
11                          duong pho
12                               None
29                          cong vien
45                             be boi
52                          nong thon
54                               vuon
76                          san trong
160                       thien nhien
161                     ho (mot phan)
162                              song
373                              bien
374                          bai bien
377                         dai duong
379             bien (huong mot phan)
382              dai duong (mot phan)
384                              cang
437                              vinh
708                                ho
1426                       thung lung
1718                       thang canh
1931                         canh dem
5583                          dam pha
6233        

In [36]:
df["views"] = df["views_str"].apply(normalize_views)

In [42]:
import ast
def extract_balcony_terrace_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "balcony-terrace":
           return True
    return None

df["balcony-terrace"] = df["room_amenities"].apply(extract_balcony_terrace_features)

In [44]:
import ast
def extract_non_smoking_room_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "non-smoking-room":
           return True
    return None

df["non-smoking-room"] = df["room_amenities"].apply(extract_non_smoking_room_features)

In [48]:
import ast
def extract_closet_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "closet":
           return True
    return None

df["closet"] = df["room_amenities"].apply(extract_closet_features)

In [50]:
import ast
def extract_air_conditioning_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "air-conditioning":
           return True
    return None

df["air_conditioning"] = df["room_amenities"].apply(extract_air_conditioning_features)

In [51]:
import ast
def extract_mini_bar_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "mini-bar":
           return True
    return None

df["mini_bar"] = df["room_amenities"].apply(extract_mini_bar_features)

In [55]:
import ast
def extract_hair_dryer_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "hair-dryer":
           return True
    return None

df["hair_dryer"] = df["room_amenities"].apply(extract_hair_dryer_features)

In [56]:
import ast
def extract_extra_long_beds_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "extra-long-beds":
           return True
    return None

df["extra-long-beds"] = df["room_amenities"].apply(extract_extra_long_beds_features)

In [57]:
import ast
def extract_complimentary_bottled_water_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "complimentary-bottled-water":
           return True
    return None

df["complimentary-bottled-water"] = df["room_amenities"].apply(extract_complimentary_bottled_water_features)

In [58]:
import ast
def extract_bathtub_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bathtub":
           return True
    return None

df["bathtub"] = df["room_amenities"].apply(extract_bathtub_features)

In [59]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "shower":
           return True
    return None

df["shower"] = df["room_amenities"].apply(extract_features)

In [60]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "separate-shower-and-tub":
           return True
    return None

df["separate-shower-and-tub"] = df["room_amenities"].apply(extract_features)

In [61]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "refrigerator":
           return True
    return None

df["refrigerator"] = df["room_amenities"].apply(extract_features)

In [62]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "high-floor":
           return True
    return None

df["high-floor"] = df["room_amenities"].apply(extract_features)

In [63]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "dressing-room":
           return True
    return None

df["dressing-room"] = df["room_amenities"].apply(extract_features)

In [64]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "ground-floor":
           return True
    return None

df["ground-floor"] = df["room_amenities"].apply(extract_features)

In [65]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "private-pool":
           return True
    return None

df["private-pool"] = df["room_amenities"].apply(extract_features)

In [66]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "executive-lounge-access":
           return True
    return None

df["executive-lounge-access"] = df["room_amenities"].apply(extract_features)

In [67]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "top-floor":
           return True
    return None

df["top-floor"] = df["room_amenities"].apply(extract_features)

In [68]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "complimentary-instant-coffee":
           return True
    return None

df["complimentary-instant-coffee"] = df["room_amenities"].apply(extract_features)

In [69]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "jacuzzi-bathtub":
           return True
    return None

df["jacuzzi-bathtub"] = df["room_amenities"].apply(extract_features)

In [70]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "smoking-allowed":
           return True
    return None

df["smoking-allowed"] = df["room_amenities"].apply(extract_features)

In [71]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "electric-blanket":
           return True
    return None

df["electric-blanket"] = df["room_amenities"].apply(extract_features)

In [72]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "free-welcome-drink":
           return True
    return None

df["free-welcome-drink"] = df["room_amenities"].apply(extract_features)

In [73]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "low-floor":
           return True
    return None

df["low-floor"] = df["room_amenities"].apply(extract_features)

In [74]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "complimentary-tea":
           return True
    return None

df["complimentary-tea"] = df["room_amenities"].apply(extract_features)

In [76]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "coffee-tea-maker":
           return True
    return None

df["coffee-tea-maker"] = df["room_amenities"].apply(extract_features)

In [78]:
import ast
def extract_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "hot-spring-access":
           return True
    return None

df["hot-spring-access"] = df["room_amenities"].apply(extract_features)

In [81]:
df.to_csv("checkpoint7.csv", index=False)

In [2]:
import pandas as pd
df = pd.read_csv("checkpoint7.csv")

In [5]:
import ast
def extract_bedroom_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "bedroom":
           return amenity.get("amenity")
    return None

df["bedroom_str"] = df["room_amenities"].apply(extract_bedroom_features)
df[["bedroom_str"]].drop_duplicates()

,bedroom_str
0,None
50,2 phòng ngủ
92,3 phòng ngủ
292,Studio/1 phòng ngủ
331,12 phòng ngủ
338,4 phòng ngủ
849,7 phòng ngủ
855,13 phòng ngủ
911,5 phòng ngủ
913,15 phòng ngủ


In [9]:
import re

def extract_bedroom_count(text):
    if not isinstance(text, str):
        return 0
    # Ưu tiên tìm số ở đầu chuỗi (ví dụ: "2 phòng ngủ", "3 phòng ngủ")
    match = re.match(r"(\d+)", text)
    if match:
        return int(match.group(1))
    # Trường hợp đặc biệt: "Studio/1 phòng ngủ" hoặc "Studio"
    if "studio" in text.lower():
        return 1
    return 0

df["bedroom_count"] = df["bedroom_str"].apply(extract_bedroom_count)

In [10]:
df[["bedroom_count","bedroom_str"]].drop_duplicates()

,bedroom_count,bedroom_str
0,0,None
50,2,2 phòng ngủ
92,3,3 phòng ngủ
292,1,Studio/1 phòng ngủ
331,12,12 phòng ngủ
338,4,4 phòng ngủ
849,7,7 phòng ngủ
855,13,13 phòng ngủ
911,5,5 phòng ngủ
913,15,15 phòng ngủ


In [11]:
df.drop(columns=["bedroom_str"])

,room_type_id,room_amenities,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,...,jacuzzi-bathtub,smoking-allowed,electric-blanket,free-welcome-drink,low-floor,complimentary-tea,coffee-tea-maker,air-bath-access,hot-spring-access,bedroom_count
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",1,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",0,1,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN,0
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",0,1,0,0,0,0,0,0,...,NaN,True,NaN,NaN,True,NaN,NaN,NaN,NaN,0
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",0,1,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",1,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,9367,[],0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
10062,9742,[],0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
10063,9743,[],0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
10064,9749,[],0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [14]:
import ast
def extract_cf_in_features(amenities):
    if isinstance(amenities, str):
        amenities = ast.literal_eval(amenities)
    for amenity in amenities:
        if isinstance(amenity, dict) and amenity.get("type") == "confirmation-instant":
           return amenity.get("amenity")
    return None

df["confirmation-instant_str"] = df["room_amenities"].apply(extract_cf_in_features)
df[["confirmation-instant_str"]].drop_duplicates()

,confirmation-instant_str
0,Điều hòa cá nhân
1,Cửa sổ
2,Hành lang ngoài
10,None
16,Cửa sổ có thể mở ra
18,Ấm nước điện
22,Đồ dùng cho giấc ngủ thoải mái
26,Rượu
87,Đồ gỗ ngoài trời
91,Trái cây/đồ ăn vặt


In [15]:
def normalize_cf_in(views_str):
    import unicodedata
    if isinstance(views_str, str):
        # Xóa "Hướng" và strip
        s = views_str.replace("Hướng", "").strip()
        # Chuyển sang không dấu
        s = unicodedata.normalize('NFKD', s)
        s = ''.join([c for c in s if not unicodedata.combining(c)])
        s = s.replace('Đ', 'D').replace('đ', 'd')
        return s.lower()
    return None
df["confirmation-instant"] = df["confirmation-instant_str"].apply(normalize_cf_in)

In [17]:
df[["confirmation-instant"]].drop_duplicates()

,confirmation-instant
0,dieu hoa ca nhan
1,cua so
2,hanh lang ngoai
10,None
16,cua so co the mo ra
18,am nuoc dien
22,do dung cho giac ngu thoai mai
26,ruou
87,do go ngoai troi
91,trai cay/do an vat


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10066 entries, 0 to 10065
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   room_type_id                  10066 non-null  int64  
 1   room_amenities                10066 non-null  object 
 2   large_double_bed              10066 non-null  int64  
 3   large_bed                     10066 non-null  int64  
 4   single_bed                    10066 non-null  int64  
 5   sofa_bed                      10066 non-null  int64  
 6   double_bed                    10066 non-null  int64  
 7   small_double_bed              10066 non-null  int64  
 8   king_size_bed                 10066 non-null  int64  
 9   futon_mattress                10066 non-null  int64  
 10  bunk_bed                      10066 non-null  int64  
 11  extra_long_bed                10066 non-null  int64  
 12  flexibility_score             10066 non-null  int64  
 13  s

In [19]:
df.drop(columns=["room_amenities"]).to_csv("final_process_room_amenities_cleaning.csv")